# Kaggle CrackTrack Reproduction Notebook

This notebook consolidates the full local reproduction workflow in one place.

It includes:
- Kaggle dataset download via `kagglehub`
- Kaggle kernel pull attempt + output fallback
- Segmentation data preparation
- YOLO segmentation training
- Evaluation and result summarization

## 1) Notebook Setup and Dependencies

Import required libraries and ensure the key packages are available.

In [ ]:
import os
import json
import shutil
import subprocess
from pathlib import Path

# Install missing runtime dependencies used in this workflow.
subprocess.run(["python", "-m", "pip", "install", "-q", "kaggle", "kagglehub"], check=True)

import kagglehub

## 2) Runtime Configuration and Constants

Define project paths, commands, and run configuration values.

In [ ]:
PROJECT_ROOT = Path.cwd()
KAGGLE_SOURCES = PROJECT_ROOT / "kaggle_sources"
KERNEL_DIR = KAGGLE_SOURCES / "cracktrack"
KERNEL_OUTPUT_DIR = KAGGLE_SOURCES / "cracktrack_output"
DATASET_LOCAL_DIR = KAGGLE_SOURCES / "concrete-crack-images"

LOCAL_REPRO_ROOT = PROJECT_ROOT / "local_repro"
RAW_SEG_DIR = LOCAL_REPRO_ROOT / "raw_segmentation"
SEG_YOLO_DIR = LOCAL_REPRO_ROOT / "segment_yolo"
RUNS_DIR = PROJECT_ROOT / "runs"

TRAIN_NAME = "cracktrack_repro"
DATASET_HANDLE = "jakubniemiec/concrete-crack-images"
KERNEL_HANDLE = "yoganandparab/cracktrack"

PROJECT_ROOT, DATASET_LOCAL_DIR

## 3) Copy Existing Utility Functions

Helpers for shell execution, path management, JSON loading, and safe symlink handling.

In [ ]:
def run_cmd(cmd, cwd=PROJECT_ROOT, check=True):
    print(f"\n$ {' '.join(cmd)}")
    result = subprocess.run(
        cmd,
        cwd=str(cwd),
        text=True,
        capture_output=True,
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {' '.join(cmd)}")
    return result


def ensure_clean_dir(path: Path):
    path.mkdir(parents=True, exist_ok=True)
    return path


def replace_path(path: Path):
    if path.is_symlink() or path.exists():
        if path.is_dir() and not path.is_symlink():
            shutil.rmtree(path)
        else:
            path.unlink()


def safe_symlink(src: Path, dst: Path):
    replace_path(dst)
    dst.symlink_to(src)


def read_json(path: Path):
    return json.loads(path.read_text())

## 4) Copy Existing Classes and Core Logic

Core pipeline functions that orchestrate the existing repo scripts in dependency order.

In [ ]:
def download_kaggle_assets():
    ensure_clean_dir(KAGGLE_SOURCES)
    ensure_clean_dir(KERNEL_DIR)

    # 1) Pull kernel source (may fail without Kaggle API credentials)
    pull_res = run_cmd(
        ["kaggle", "kernels", "pull", KERNEL_HANDLE, "--path", str(KERNEL_DIR)],
        check=False,
    )
    if pull_res.returncode != 0:
        print("Kernel pull failed (likely missing Kaggle API auth). Continuing with output fallback.")

    # 2) Download notebook outputs as fallback
    output_path = Path(kagglehub.notebook_output_download(KERNEL_HANDLE))
    replace_path(KERNEL_OUTPUT_DIR)
    shutil.copytree(output_path, KERNEL_OUTPUT_DIR)

    # 3) Download dataset
    dataset_cache_path = Path(kagglehub.dataset_download(DATASET_HANDLE))
    replace_path(DATASET_LOCAL_DIR)
    shutil.copytree(dataset_cache_path, DATASET_LOCAL_DIR)

    return {
        "kernel_pull_code": pull_res.returncode,
        "kernel_output_path": str(KERNEL_OUTPUT_DIR),
        "dataset_path": str(DATASET_LOCAL_DIR),
    }


def prepare_segmentation_data():
    ensure_clean_dir(LOCAL_REPRO_ROOT)
    ensure_clean_dir(RAW_SEG_DIR)

    images_src = DATASET_LOCAL_DIR / "images"
    masks_src = DATASET_LOCAL_DIR / "masks"
    if not images_src.exists() or not masks_src.exists():
        raise FileNotFoundError("Dataset images/masks not found in local Kaggle dataset copy.")

    safe_symlink(images_src.resolve(), RAW_SEG_DIR / "images")
    safe_symlink(masks_src.resolve(), RAW_SEG_DIR / "masks")

    run_cmd([
        "python", "src/prepare_segment_data.py",
        "--data_root", "local_repro",
        "--output_dir", "local_repro/segment_yolo",
        "--seed", "42",
    ])


def train_and_evaluate():
    run_cmd([
        "python", "src/train_yolo_segment.py",
        "--data_yaml", "local_repro/segment_yolo/crack-seg.yaml",
        "--runs_dir", "./runs",
        "--model", "yolov8n-seg.pt",
        "--epochs", "3",
        "--batch", "8",
        "--imgsz", "256",
        "--patience", "3",
        "--workers", "2",
        "--name", TRAIN_NAME,
        "--exist_ok",
    ])

    run_cmd([
        "python", "src/eval_segment.py",
        "--model", f"runs/segment/{TRAIN_NAME}/weights/best.pt",
        "--data_yaml", "local_repro/segment_yolo/crack-seg.yaml",
        "--split", "test",
        "--imgsz", "256",
        "--save_worst", "10",
        "--save_best", "5",
    ])

## 5) Input Loading and Preprocessing

Download Kaggle assets and prepare YOLO segmentation dataset structure from `images/` and `masks/`.

In [ ]:
download_info = download_kaggle_assets()
prepare_segmentation_data()
download_info

## 6) Main Execution Pipeline

Train YOLO segmentation and run test-set evaluation.

In [ ]:
train_and_evaluate()

## 7) Output Validation and Quick Tests

Verify that expected files exist and print core quality metrics.

In [ ]:
metrics_path = RUNS_DIR / "segment" / TRAIN_NAME / "eval" / "metrics_test.json"
best_model_path = RUNS_DIR / "segment" / TRAIN_NAME / "weights" / "best.pt"

assert SEG_YOLO_DIR.exists(), "Prepared segmentation directory missing"
assert best_model_path.exists(), "Best model checkpoint not found"
assert metrics_path.exists(), "Evaluation metrics JSON not found"

metrics = read_json(metrics_path)
summary = {
    "num_images": metrics["num_images"],
    "dice_mean": metrics["dice"]["mean"],
    "iou_mean": metrics["iou"]["mean"],
}
summary

## 8) Save Results and Artifacts

Persist a compact reproduction summary and list key output locations.

In [ ]:
artifact_manifest = {
    "project_root": str(PROJECT_ROOT),
    "dataset_local_dir": str(DATASET_LOCAL_DIR),
    "kernel_dir": str(KERNEL_DIR),
    "kernel_output_dir": str(KERNEL_OUTPUT_DIR),
    "segment_data_yaml": str(SEG_YOLO_DIR / "crack-seg.yaml"),
    "best_model": str(best_model_path),
    "eval_metrics": str(metrics_path),
    "summary": summary,
}

summary_path = KAGGLE_SOURCES / "cracktrack_repro_summary.json"
summary_path.parent.mkdir(parents=True, exist_ok=True)
summary_path.write_text(json.dumps(artifact_manifest, indent=2))

print(f"Saved summary: {summary_path}")
artifact_manifest